In [30]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import joblib


# 1. LOAD
df = pd.read_csv("Heart_Dataset.csv")
df = df.drop(["FastingBS","RestingECG"], axis=1)

X = df.drop(["HeartDisease"], axis=1)
print(X.columns)
y = df["HeartDisease"]
print(X)
print(X["ST_Slope"].unique())

numerical = ["Age", "RestingBP", "Cholesterol", "MaxHR","Oldpeak"]
categorical = ["Sex", "ChestPainType", "ExerciseAngina","ST_Slope"]

# 2. CLEAN NUMERICAL:
for col in numerical:
    X[col] = pd.to_numeric(X[col], errors='coerce') # converts " to NaN
    X[col] = X[col].replace(0, np.nan) # 0 cholesterol is invalid
X[numerical] = X[numerical].fillna(X[numerical].mean())

# 3. CLEAN CATEGORICAL:
for col in categorical:
    X[col] = X[col].fillna(X[col].mode()[0])

# 4. SCALE NUMERICAL
scaler = StandardScaler()
num_scaled = pd.DataFrame(
    scaler.fit_transform(X[numerical]),
    columns=numerical,
    index=X.index
)

# 5. ENCODE CATEGORICAL
cat_encoded = pd.get_dummies(X[categorical], drop_first=True).astype(int)

# 6. COMBINE
X= pd.concat([num_scaled, cat_encoded], axis=1)

print("Final X shape:", X.shape)
print("Any NaN left?", X.isnull().sum().sum()) # MUST be 0
print(X.head())


Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'MaxHR',
       'ExerciseAngina', 'Oldpeak', 'ST_Slope'],
      dtype='str')
     Age Sex ChestPainType  RestingBP  Cholesterol  MaxHR ExerciseAngina  \
0     40   M           ATA        140          289    172              N   
1     49   F           NAP        160          180    156              N   
2     37   M           ATA        130          283     98              N   
3     48   F           ASY        138          214    108              Y   
4     54   M           NAP        150          195    122              N   
..   ...  ..           ...        ...          ...    ...            ...   
913   45   M            TA        110          264    132              N   
914   68   M           ASY        144          193    141              N   
915   57   M           ASY        130          131    115              Y   
916   57   F           ATA        130          236    174              N   
917   38   M         

In [31]:
# q2) TRAIN
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("X_train shape : ",X_train.shape)
print("X_test shape : ",X_test.shape)
print("y_train shape : ",y_train.shape)
print("y_test shape : ",y_test.shape)



X_train shape :  (734, 12)
X_test shape :  (184, 12)
y_train shape :  (734,)
y_test shape :  (184,)


In [32]:
model = LogisticRegression()
pred = model.fit(X_train,y_train)


In [33]:
y_pred=model.predict(X_test)
print("Predicted values : ")
print(y_pred[0:10])
print("\nActual values : ")
print(y_test[0:10])




Predicted values : 
[0 1 1 1 0 1 1 0 1 1]

Actual values : 
668    0
30     1
377    1
535    1
807    0
793    1
363    1
583    0
165    1
483    1
Name: HeartDisease, dtype: int64


In [34]:
con_mat = confusion_matrix(y_true=y_test,y_pred=y_pred)
print("Confusion Matrix : ")
print(con_mat)
print("TN : ",con_mat[0][0])
print("FP : ",con_mat[0][1])
print("FN : ",con_mat[1][0])
print("TP : ",con_mat[1][1])

Confusion Matrix : 
[[67 10]
 [20 87]]
TN :  67
FP :  10
FN :  20
TP :  87


In [35]:
report = classification_report(y_true= y_test,y_pred=y_pred)
print("The classification report : ")
print(report)

The classification report : 
              precision    recall  f1-score   support

           0       0.77      0.87      0.82        77
           1       0.90      0.81      0.85       107

    accuracy                           0.84       184
   macro avg       0.83      0.84      0.84       184
weighted avg       0.84      0.84      0.84       184



In [ ]:
joblib.dump(model, "heart_model.pkl")
joblib.dump(scaler, "heart_scaler.pkl") 
joblib.dump(X.columns.tolist(), "heart_columns.pkl")
print(type(scaler))

<class 'sklearn.preprocessing._data.StandardScaler'>
